In [ ]:
import pandas as pd
import polars as pl
from pathlib import Path

In [ ]:
# Analyze all parquet files in this folder using polars

# Directory containing this notebook and the parquet files
base_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

parquet_files = sorted(base_dir.glob("*.parquet"))

if not parquet_files:
    print("No parquet files found in", base_dir)
else:
    for path in parquet_files:
        print(f"\n=== {path.name} ===")
        try:
            df = pl.read_parquet(path)
        except Exception as e:
            print(f"  Failed to read file: {e}")
            continue

        if df.is_empty():
            print("  File is empty.")
            continue

        # Heuristically detect date and symbol columns
        lower_cols = {c.lower(): c for c in df.columns}

        date_col = None
        for candidate in ("date", "timestamp", "trade_date", "datetime"):
            if candidate in lower_cols:
                date_col = lower_cols[candidate]
                break

        symbol_col = None
        for candidate in ("symbol", "ticker", "instrument", "secid"):
            if candidate in lower_cols:
                symbol_col = lower_cols[candidate]
                break

        if date_col is None:
            print("  Could not identify a date column.")
        else:
            date_stats = df.select([
                pl.col(date_col).min().alias("start_date"),
                pl.col(date_col).max().alias("end_date"),
            ]).to_dicts()[0]
            print(f"  Date range: {date_stats['start_date']} -> {date_stats['end_date']}")

        if symbol_col is None:
            print("  Could not identify a symbol column.")
        else:
            n_symbols = df.select(pl.col(symbol_col).n_unique().alias("n_symbols")).to_dicts()[0]["n_symbols"]
            print(f"  Number of symbols: {n_symbols}")